# HTB SMS Spam Collection: download, inspect, deduplicate

Module 292, Section 10. Use the **AI** kernel. Data source: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms%2Bspam%2Bcollection), Almeida, Hidalgo, and Yamakami (CC BY 4.0). This notebook inspects the dataset **only**. It does not tokenize, vectorize, split, or train a classifier; those steps belong to later lessons. No downloaded text or runtime output is committed to Git.

## Open the official ZIP without extracting files

HTB uses `requests.get`, `ZipFile(BytesIO(...))`, then `extractall`. Here `SMS_SPAM_ZIP` can point to a previously downloaded ZIP; otherwise the same UCI link is fetched. We enforce a modest compressed-file and member-size bound, look up only `SMSSpamCollection`, and read it in memory. `ham` means a legitimate message; `spam` means an unwanted message. A label is the answer we will eventually try to predict, but no model is trained in this section.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import csv
import os
import pandas as pd
import requests

source_url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
local_zip = Path(os.environ.get('SMS_SPAM_ZIP', 'sms+spam+collection.zip'))
if local_zip.is_file():
    archive_bytes = local_zip.read_bytes()
else:
    response = requests.get(source_url, timeout=30)
    response.raise_for_status()  # HTTP errors are not valid ZIP files.
    archive_bytes = response.content
if len(archive_bytes) > 5_000_000:
    raise ValueError('Unexpectedly large SMS archive')
with ZipFile(BytesIO(archive_bytes)) as archive:
    member = archive.getinfo('SMSSpamCollection')
    if member.file_size > 2_000_000:
        raise ValueError('Unexpectedly large SMS text member')
    with archive.open(member) as source:
        # SMS text includes literal quotes. Disable CSV quoting so two physical
        # message lines are not merged into one logical CSV record.
        raw_sms = pd.read_csv(source, sep='\t', header=None,
                              names=['label', 'message'], quoting=csv.QUOTE_NONE)
print('Raw SMS rows:', len(raw_sms))
assert list(raw_sms.columns) == ['label', 'message']
assert raw_sms['label'].isin(['ham', 'spam']).all()
if member.file_size == 477_907:  # The current course-linked UCI archive.
    assert len(raw_sms) == 5_574

## Inspect missingness and class balance

`head()` shows a sample; `info()` prints column type and non-null counts. `isnull().sum()` (say **is-null, then sum**) counts True missing indicators per column. `value_counts()` shows how many ham and spam rows we have; those are *counts*, not probabilities yet. `describe(include='all')` includes categorical/text summaries. The UCI file may have a different ham/spam balance from actual deployment traffic, so do not infer real-world prevalence from this classroom corpus.

In [ ]:
print(raw_sms.head())
raw_sms.info()  # Already prints; print(raw_sms.info()) would also print None.
print(raw_sms.describe(include='all'))
print('Missing values by column:', raw_sms.isnull().sum().to_dict())
print('Raw label counts:', raw_sms['label'].value_counts().to_dict())
assert raw_sms[['label', 'message']].notna().all().all()

## Count and remove **exact** duplicates

`duplicated()` marks a row True when its label **and** message repeat an earlier row; summing True values gives the number of extra copies. `drop_duplicates()` keeps the first copy of each exact row. A similar-looking phishing message with a changed phone number is not exact, and conflicting labels for the same text require separate investigation. Deduplicate before any later split so identical messages cannot leak across train/test folds. Keep `raw_sms` unchanged for audit.

In [ ]:
duplicate_count = int(raw_sms.duplicated().sum())
unique_sms = raw_sms.drop_duplicates().reset_index(drop=True)
print('Extra exact copies:', duplicate_count)
print('Unique label/message rows:', len(unique_sms))
print('Label counts after deduplication:', unique_sms['label'].value_counts().to_dict())
assert len(unique_sms) + duplicate_count == len(raw_sms)
assert not unique_sms.duplicated().any()
if len(raw_sms) == 5_574:
    assert duplicate_count == 403 and len(unique_sms) == 5_171

We now have a labeled, inspected, exact-deduplicated SMS table. This is **not** a trained spam detector. The notebook deliberately has no text vectorizer, model fit, held-out evaluation, or decision threshold; those must be introduced in their own HTB sections. The downloaded ZIP remains outside Git.